# End-to-end pipeline

Runs the project modules in order. Calculations live in `src/epo_project`; this notebook only calls them and shows key outputs.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
import pandas as pd
from epo_project import config, data, replication, simulation

## Steps 1–2: industry returns, momentum and risk inputs

Downloads the French files (reused if already present), writes `data/processed` and `data/interim`, and checks the completion criteria.

In [2]:
summary = data.prepare_returns()
{k: v for k, v in summary.items() if k != "industries"}

{'first_month': '1926-07-31',
 'last_common_month': '2026-08-31',
 'missing_industry_months': 2868,
 'rf_source': 'F-F_Research_Data_Factors (monthly RF, one-month T-bill)',
 'eligibility_rule': 'momentum_ready: >= 12 consecutive months ending at t; risk_ready: >= 60; benchmark_eligible: both'}

In [3]:
data.build_inputs()

{'formation_dates': 1143,
 'first': '1931-06-30',
 'last': '2026-08-31',
 'industries_min_max': (40, 49),
 'smallest_eigenvalue_q': 3.545648451161342e-05}

## Step 3: Equity 1 benchmark validation

Raw MVO, MVO with Q and scalar EPO on the proposal grid, each scaled to 10% predicted volatility; w chosen each January from the preceding 60 months of Sharpe ratios. Published Equity 1 values are shown for comparison.

In [4]:
bench = replication.run()
bench["table"].round(3)

,portfolio,sharpe,sharpe_published,months,difference
0,Raw MVO (Sigma_hat),0.303,0.19,924,0.113
1,MVO with Q (EPO w=0),0.572,0.56,924,0.012
2,EPO w=0.25,0.720,0.75,924,-0.030
3,EPO w=0.50,0.770,0.79,924,-0.020
4,EPO w=0.75,0.796,0.80,924,-0.004
5,EPO w=0.90,0.804,0.79,924,0.014
6,EPO w=0.95,0.803,NaN,924,NaN
7,EPO w=0.99,0.794,0.73,924,0.064
8,OOS EPO,0.751,0.79,924,-0.039


In [5]:
bench["diagnostics"].round(3)

,mean_eigenvalue,realized_to_predicted_vol,ex_ante_sharpe_relative,realized_sharpe_signal_direction,risk_share_mvo_q,risk_share_oos_epo
eigenvalue_group,,,,,,
1,0.068,1.948,0.964,0.140,0.190,0.035
2,0.127,1.569,1.020,0.014,0.208,0.046
3,0.253,1.235,1.016,0.321,0.201,0.071
4,0.516,0.964,1.013,0.573,0.204,0.169
5,4.344,0.881,0.986,0.542,0.198,0.679


## Step 4: scalar compression loss

Known-covariance Gaussian model with 49 assets and 1,000 paired draws per heterogeneity level. Compares the heterogeneous allocation with the best scalar (oracle) and the mean-uncertainty scalar on the same signals; uniform `h` must give zero loss. Losses are in basis points of monthly utility.

In [ ]:
sim = simulation.run()
sim["audit"]

In [ ]:
sim["table"].round(3)